# Solar System Visualisation

In [37]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from skyfield.api import load

# load planetary ephemeris data
eph = load('de440.bsp')  # JPL ephemeris file
planets = {
    'Mercury': eph['mercury'],
    'Venus': eph['venus'],
    'Earth': eph['earth'],
    'Mars': eph['mars barycenter'],
    'Jupiter': eph['jupiter barycenter'],
    'Saturn': eph['saturn barycenter'],
    'Uranus': eph['uranus barycenter'],
    'Neptune': eph['neptune barycenter'],
}

# define the time range
ts = load.timescale()
start_date = ts.utc(1993, 12, 11)
end_date = ts.utc(2024, 12, 11)
time_range = ts.tt_jd(np.linspace(start_date.tt, end_date.tt, 30 * (2024 - 1993 + 1)))

In [38]:
# calculate positions
positions = {name: [] for name in planets}
for t in time_range:
    for name, planet in planets.items():
        position = planet.at(t).ecliptic_position().au[:2] 
        positions[name].append(position)

# convert positions to numpy arrays for easier handling
for name in positions:
    positions[name] = np.array(positions[name])


In [47]:
# set up the animation
fig, ax = plt.subplots(figsize=(13, 13))
fig.patch.set_facecolor('black')
ax.set_xlim(-35, 35)
ax.set_ylim(-35, 35)
ax.set_aspect('equal')
ax.axis('off')  
plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

# set planet colors and sizes
planet_colors = {
    'Sun': 'yellow',
    'Mercury': 'gray',
    'Venus': 'orange',
    'Earth': 'blue',
    'Mars': 'red',
    'Jupiter': 'brown',
    'Saturn': 'tan',
    'Uranus': 'cyan',
    'Neptune': 'blue',
}
planet_sizes = {
    'Sun': 10,
    'Mercury': 1,
    'Venus': 6,
    'Earth': 8,
    'Mars': 6,
    'Jupiter': 20,
    'Saturn': 15,
    'Uranus': 10,
    'Neptune': 10,
}
lines = {name: ax.plot([], [], 'o', label=name, color=planet_colors[name], markersize=planet_sizes[name])[0] for name in planets}

# add a white line around each planet
for name, line in lines.items():
    line.set_markeredgecolor('white')
    line.set_markeredgewidth(1)

# add legend and style it (white text on black background)
# remove the border around the legend
legend = ax.legend(loc='lower left', fontsize=20)
legend.get_frame().set_facecolor('black')
legend.get_frame().set_linewidth(0)
for text in legend.get_texts():
    text.set_color('white')
    text.set_fontsize(25)
    text.set_fontname('monospace')


# add date annotation
date_text = ax.text(-33, 33, '', color='orange', fontsize=25, ha='left', va='center', fontname='monospace')


def update(frame):
    if frame >= len(time_range):
        frame = len(time_range) - 1  # use the last frame during the pause
    for name, line in lines.items():
        # get only the current position
        x, y = positions[name][frame].T
        line.set_data([x], [y])  # set only the current position
    date_text.set_text(str(time_range[frame].utc_strftime('%Y-%m-%d')))
    return [*lines.values(), date_text]


pause_duration = 30
total_frames = len(time_range) + pause_duration
ani = FuncAnimation(
    fig, update, frames=total_frames, blit=True, interval=30
)
ani.save('solar_system_animation.gif', writer='imagemagick')


![](solar_system_animation.gif)

In [48]:
# Set up the animation
fig, ax = plt.subplots(figsize=(13, 13))
fig.patch.set_facecolor('black')
ax.set_xlim(-35, 35)
ax.set_ylim(-35, 35)
ax.set_aspect('equal')
ax.axis('off') 
plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

lines = {name: ax.plot([], [], 'o', label=name, color=planet_colors[name], markersize=planet_sizes[name])[0] for name in planets}

legend = ax.legend(loc='lower left', fontsize=20)
legend.get_frame().set_facecolor('black')
legend.get_frame().set_linewidth(0)
for text in legend.get_texts():
    text.set_color('white')
    text.set_fontsize(25)
    text.set_fontname('monospace')

date_text = ax.text(-33, 33, '', color='orange', fontsize=25, ha='left', va='center', fontname='monospace')

def update_with_traces(frame):
    if frame >= len(time_range):
        frame = len(time_range) - 1  
    for name, line in lines.items():
        x, y = positions[name][:frame + 1].T
        line.set_data(x, y)
    date_text.set_text(str(time_range[frame].utc_strftime('%Y-%m-%d')))
    return [*lines.values(), date_text]

ani = FuncAnimation(
    fig, update_with_traces, frames=total_frames, blit=True, interval=30
)
ani.save('solar_system_animation_with_traces.gif', writer='imagemagick')

![](solar_system_animation_with_traces.gif)